# 04 · Validation

The overfitting question, answered with numbers: fit on train, report on test.

Whatever this notebook returns is what the slide says.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np, pandas as pd
pd.set_option('display.width', 140)

# Synthetic market today. When Matt's A3 cache exists, this becomes:
#     from quant.factors.base import load_panel
#     panel = load_panel()
from tests.conftest import make_synthetic_panel
panel, truth = make_synthetic_panel()
panel


In [ ]:
from quant.signals.generation import SignalEngine

engine = SignalEngine.default().fit(panel, train_frac=0.6, method='ic')
train, test = engine.train_test_dates(train_frac=0.6)
print(f'TRAIN {train[0].date()}..{train[-1].date()}  n={len(train)}')
print(f'TEST  {test[0].date()}..{test[-1].date()}  n={len(test)}')
print(f'embargo gap: {(test[0]-train[-1]).days} calendar days')


## The fitted weights

This cell is the answer to 'where did the weights come from?'


In [ ]:
print(engine.fitted)
engine.fitted.diagnostics[['mean_ic','t_stat','raw_strength','weight']].round(4)


## In-sample vs out-of-sample

The gap between these two IS the overfitting measure. A large drop means
the fit found noise.


In [ ]:
ins = engine.evaluate(panel, dates=train, label='in-sample')
oos = engine.evaluate(panel, dates=test,  label='out-of-sample')
for r in (ins, oos):
    print(r)


## Against the baseline

If the fitted model cannot beat equal weights out of sample, ship equal weights.


In [ ]:
baseline = SignalEngine.default().use_equal_weights(panel)
base_oos = baseline.evaluate(panel, dates=test, label='equal-weight')
print(oos)
print(base_oos)
print()
print(f'fitted beats baseline OOS: {oos.mean_ic > base_oos.mean_ic}')


## Does it depend on the regime?

B14. A factor that only works in one regime is a factor you need to know
the regime for.


In [ ]:
from quant.signals.regime import classify_regimes, ic_by_regime, regime_summary
regimes = classify_regimes(panel)
print(regime_summary(regimes))
print()
ic_by_regime(engine.panels['momentum_12_1'], panel, regimes, name='momentum_12_1', min_periods=4)[['n','mean_ic','t_stat']]
